# 네이버 영화 댓글 감성분석

- nsmc(Naver Sentiment Movie Corpus) dataset
- 입력: 영화에 대한 댓글
- 출력: 부정(0)/긍정(1)
- 구글에서 nsmc로 조회(https://github.com/e9t/nsmc)

In [1]:
!uv pip install requests pandas scikit-learn==1.8 kiwipiepy

Using Python 3.12.13 environment at: c:\Documents\SKN31-inst\08_NLP\.venv
Checked 4 packages in 663ms


In [5]:
# 데이터셋 다운로드 및 DataFrame 생성
import requests
import pandas as pd
from io import StringIO # (메모리에 있는) String과 입출력 작업을 할 수 있게 하는 객체.

url = "https://raw.githubusercontent.com/e9t/nsmc/refs/heads/master/ratings.txt"
resp = requests.get(url)
if resp.status_code == 200:
    str_io = StringIO(resp.text)

In [6]:
df = pd.read_csv(str_io, sep="\t")
df.shape

(200000, 3)

In [7]:
df.head()

,id,document,label
0,8112052,어릴때보고 지금다시봐도 재밌어요ㅋㅋ,1
1,8132799,"디자인을 배우는 학생으로, 외국디자이너와 그들이 일군 전통을 통해 발전해가는 문화산...",1
2,4655635,폴리스스토리 시리즈는 1부터 뉴까지 버릴께 하나도 없음.. 최고.,1
3,9251303,와.. 연기가 진짜 개쩔구나.. 지루할거라고 생각했는데 몰입해서 봤다.. 그래 이런...,1
4,10067386,안개 자욱한 밤하늘에 떠 있는 초승달 같은 영화.,1


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        200000 non-null  int64
 1   document  199992 non-null  str  
 2   label     200000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 4.6 MB


In [ ]:
# 결측치 제거
df = df.dropna()
df.isnull().sum()

id          0
document    0
label       0
dtype: int64

In [10]:
df.shape

(199992, 3)

In [ ]:
# 중복(document)행을 확인/제거
df[df.duplicated(subset=['document'], keep=False)].sort_values("document")
# keep=False: 모든 중복행들을 True로 반환.
# keep="first" | "last" :  중복 된 것들에서 first: 첫번째, last: 마지막 것만 True (중복된 행의 값만 확인할 때)

,id,document,label
136580,6993402,!,0
12986,181912,!,1
59493,7448690,",",1
15611,7868198,",",1
101742,7481337,",,,",0
...,...,...,...
1961,5158304,힐러리 더프의 매력에 빠지다!!!,1
104378,3010576,힘내세요,0
152847,4052413,힘내세요,0
118102,7024515,힘들다,0


In [18]:
# 중복행을 삭제 - 하나만 남기고 나머지는 삭제
df = df.drop_duplicates(subset=['document'])

In [20]:
print(df.shape)
df[df.duplicated(subset=['document'], keep=False)].sort_values("document")

(194543, 3)


,id,document,label


In [23]:
# index name 초기화
df.reset_index(drop=True, inplace=True)

In [24]:
# ------------------------------------------------
# 댓글 데이터셋 제공 class
## 1. 데이터 다운로드 및 저장
## 2. 기본적인 전처리
## 3. 형태소 기반 토큰화 진행

In [31]:
import os
import requests
import pandas as pd
from kiwipiepy import Kiwi


class NSMCTokenizer:
    
    def __init__(self):
        # 데이터셋을 저장할 파일 경로
        os.makedirs("data", exist_ok=True)
        self.data_file_path = "data/ratings.txt"
        # Kiwi객체
        self.kiwi = Kiwi()

        # 토큰화 끝난 Dataset의 DataFrame을 속성으로 저장.
        ## 1. Dataset download and loading
        df = self.load_nsmc_dataset()

        ## 2. 댓글 전처리 + 토큰화
        self.nsmc_df = self.preprocess(df)

    def load_nsmc_dataset(self) -> pd.DataFrame:
        """
        nsmc 데이터셋을 다운받아서 data/ratings.txt로 저장하고 
        DataFrame으로 만들어서 반환.
        Args:
        Returns:
            pd.DataFrame: 네이버 영화댓글 데이터셋. 
                          columns: id(댓글번호), document(댓글내용), label(0:부정, 1:긍정)
        Raise:
            Exception: DataFrame생성과 데이터셋 다운로드 실패 시 발생.
        """
        try:
            # os.path.exist(경로)
            df =  pd.read_csv(self.data_file_path, sep="\t", encoding="utf-8")
        except:
            # 다운로드 -> 저장 -> DF 생성
            url = "https://raw.githubusercontent.com/e9t/nsmc/refs/heads/master/ratings.txt"
            resp = requests.get(url)
            if resp.status_code == 200:
                # 다운로드 및 저장
                with open(self.data_file_path, "wt", encoding="utf-8") as f:
                    f.write(resp.text)
                # DataFrame 생성
                df =  pd.read_csv(self.data_file_path, sep="\t", encoding="utf-8")
            else:
                # 예외를 발생
                raise Exception(f"데이터파일을 다운받지 못했습니다. 에러코드: {resp.status_code}")

        return df


    def tokenize(self, doc: str) -> str:
        """
        하나의 댓글을 받아서 전처리(띄어쓰기 교정)한 뒤 토큰화한다.
        토큰화 한 것을 다시 하나의 문자열로 만들어서 반환.
        
        Args:
            doc (str) - 토큰화할 댓글(document)
        Returns:
            str - 처리결과. (띄어쓰기 교정, 토큰화)

        """
        # 띄어쓰기 교정
        doc = self.kiwi.space(doc)
        token_list = [] # 토큰(원형)들 저장
        try:
            for token in self.kiwi.tokenize(doc): # list[Token]
                token_list.append(token.lemma)
        except:
            pass
        
        return " ".join(token_list)

    def preprocess(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        전체 Dataset을 전처리(결측치 제거, 중복(document)행 제거), 개별 데이터 토큰화작업
        Args:
            df (pd.DataFrame) - 전처리 및 토큰화 대상 데이터셋.
        Returns:
            pd.DataFrame - 처리결과.
        """
        # 결측치 행 제거
        result_df = df.dropna()
        # document 컬럼 중복행 제거
        result_df = result_df.drop_duplicates(subset=['document'])

        # document (댓글) 컬럼의 값들을 토큰화 처리.
        result_df['document'] = result_df['document'].apply(self.tokenize)

        # document의 글자수가 0인 행들을 제거(불용어 제거등 전처리 후 글자수가 0이 될 수 있다.)
        drop_row_index = result_df[result_df['document'].str.len() == 0].index
        result_df = result_df.drop(index=drop_row_index)

        return result_df

In [34]:
import time
s = time.time()

nsmc = NSMCTokenizer()

print(f"걸린시간: {time.time() - s}초")

걸린시간: 145.85836100578308초


In [36]:
nsmc.nsmc_df.shape
nsmc.nsmc_df.head()

,id,document,label
0,8112052,어리다 ᆯ 때 보다 고 지금 다시 보다 어도 재밌다 어요 ㅋㅋ,1
1,8132799,"디자인 을 배우다 는 학생 으로 , 외국 디자이너 와 그 들 이 일구다 ᆫ 전통 을...",1
2,4655635,폴리스 스토리 시리즈 는 1 부터 뉴 까지 버리다 ᆯ께 하나 도 없다 음 .. 최고 .,1
3,9251303,와 .. 연기 가 진짜 개 쩔다 구나 .. 지루 하 ᆯ 거 이다 라고 생각 하 었 ...,1
4,10067386,안개 자욱 하 ᆫ 밤하늘 에 뜨다 어 있다 는 초승달 같다 은 영화 .,1


In [37]:
nsmc.tokenize("어제 짜장면을 중국집에서 먹었습니다.")

'어제 짜장면 을 중국집 에서 먹다 었 습니다 .'

# 모델링
- 데이터 준비
- 모델 생성
- 학습

In [38]:
# 데이터준비
## X, y 분리
X = nsmc.nsmc_df['document']
y = nsmc.nsmc_df['label']

X.shape, y.shape

((194543,), (194543,))

In [39]:
# y의 분포
y.value_counts()

label
0    97277
1    97266
Name: count, dtype: int64

In [41]:
## Train/Test/Validation set으로 분리.
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train
)

X_train.shape, X_test.shape, X_val.shape

((124507,), (38909,), (31127,))

In [51]:
# 모델 생성
# 전처리 - TfidfVectorizer, 모델: LogisticRegression -> Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline 
from sklearn.metrics import accuracy_score, classification_report

tfidf = TfidfVectorizer(
    ngram_range=(1, 2)
) # tuning: ngram_range, max_df, min_df
model = LogisticRegression()
steps = [
    ("tfidf", tfidf), ("model", model)
]
pipeline = Pipeline(steps=steps, verbose=True)


In [52]:
# 학습
pipeline.fit(X_train, y_train)

[Pipeline] ............. (step 1 of 2) Processing tfidf, total=   2.8s
[Pipeline] ............. (step 2 of 2) Processing model, total=   3.4s


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [53]:
# 검증
pred_train = pipeline.predict(X_train)
pred_val = pipeline.predict(X_val)

print(
    accuracy_score(y_train, pred_train),
    accuracy_score(y_val, pred_val)
)
# 0.8611001791063956 0.8319465415876891

0.9101094717566081 0.8477206283933563


In [54]:
pipeline.score(X_train,y_train), pipeline.score(X_val, y_val) 
# 분류모델.score(X, y) : 정확도 계산, 
# 회귀모델.score(X, y) : R2

(0.9101094717566081, 0.8477206283933563)

In [55]:
pipeline.score(X_test, y_test)

0.8497262844072065

In [56]:
# classification report
print(classification_report(y_val, pred_val))

              precision    recall  f1-score   support

           0       0.84      0.85      0.85     15564
           1       0.85      0.84      0.85     15563

    accuracy                           0.85     31127
   macro avg       0.85      0.85      0.85     31127
weighted avg       0.85      0.85      0.85     31127



In [59]:
# 새로운 데이터를 추론
def predict(*comment):
    # NSMC 를 이용해서 token화
    pre_comment = [nsmc.tokenize(doc) for doc in comment]
    pred = pipeline.predict(pre_comment)
    return pred

In [66]:
result = predict(
    "이영화 진짜 별로다.", "시간 때우려고 봤는데 엄청 재미있다.", "배우들 연기가 미쳤다.",
    "내 소중한 시간 돌려줘.", "여기 식당 인테리어가 좋아요.", "음식이 맛있다.", "배송이 빨라요."
)

In [67]:
import numpy as np
idx_to_class = np.array(['부정적 댓글', '긍정적 댓글'])
print(result, idx_to_class[result])

[0 1 1 0 1 1 1] ['부정적 댓글' '긍정적 댓글' '긍정적 댓글' '부정적 댓글' '긍정적 댓글' '긍정적 댓글' '긍정적 댓글']


In [68]:
vocab = tfidf.get_feature_names_out()
vocab.shape

(479491,)

In [71]:
v = tfidf.transform(["이영화 진짜 별로다."])
# v.toarray()
print(v)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2 stored elements and shape (1, 479491)>
  Coords	Values
  (0, 355231)	0.9292018014949639
  (0, 412830)	0.36957274263467244
